# JADCO / Collection Équinoxe — Phase 3 : audit des données

Inspection agrégée des CSV locaux confidentiels. Aucun enregistrement brut, adresse, URL, identifiant d’unité ou export individuel n’est affiché. Les fichiers bruts sont lus uniquement. Aucun nettoyage, ingénierie de variables ou prévision n’est effectué. Les résumés catégoriels se limitent aux champs descriptifs pertinents ; la couverture des identifiants est présentée sous forme de dénombrements.

Exécuter depuis le répertoire `notebooks/`. `UNIT_KEY` est la paire officielle `(sPropCode, sUnitCode)`. Un tuple est utilisé en mémoire pour éviter une concaténation ambiguë ; aucun jeu de données dérivé n’est enregistré. Les composantes manquantes de la clé sont comptées séparément.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display as ipython_display, Markdown

def display(value):
    # Traduire uniquement les libellés affichés ; conserver les résultats calculés.
    if isinstance(value, pd.DataFrame):
        value = value.rename(columns=DISPLAY_LABELS).rename_axis(
            index=DISPLAY_LABELS.get(value.index.name, value.index.name))
    ipython_display(value)

DISPLAY_LABELS = {'dataset': 'jeu de données', 'rows': 'lignes', 'columns': 'colonnes', 'matches_challenge_dimensions': 'dimensions conformes au défi', 'missing_count': 'valeurs manquantes', 'missing_pct': 'valeurs manquantes (%)', 'whitespace_only_count': 'valeurs composées d’espaces', 'unique_nonmissing': 'valeurs uniques non manquantes', 'category': 'catégorie', 'count': 'effectif', 'mean': 'moyenne', 'std': 'écart type', 'nonmissing_unparseable': 'valeurs non manquantes non convertibles', 'zero_count': 'valeurs nulles', 'negative_count': 'valeurs négatives', 'column': 'colonne', 'missing': 'valeurs manquantes', 'unparseable_nonmissing': 'échecs de conversion non manquants', 'years': 'années', 'missing_official_key_rows': 'lignes à clé officielle incomplète', 'unique_UNIT_KEY': 'UNIT_KEY uniques', 'duplicate_UNIT_KEY_excess_rows': 'lignes excédentaires de UNIT_KEY dupliquée', 'rows_in_duplicate_UNIT_KEY_groups': 'lignes dans les groupes de UNIT_KEY dupliquée', 'missing_site_test_components': 'lignes à composantes site/unité manquantes', 'colliding_site_unit_groups': 'groupes site/unité en collision', 'excess_official_pairs_under_site_key': 'paires officielles excédentaires sous la clé site', 'rows_affected_by_site_collisions': 'lignes concernées par les collisions site', 'days_between_observations': 'jours entre observations', 'calendar_month_gap': 'écart en mois calendaires', 'observations_per_unit': 'observations par unité', 'year': 'année', 'observations': 'observations'}

RAW = Path('../data/raw')
FILES = {name: RAW / f'equinoxe_{name}.csv' for name in
         ['listings', 'lease_history', 'concessions', 'asking_history']}
raw_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in FILES.items()}
# Préserver les identifiants (y compris les zéros initiaux) ; les autres dtypes sont inférés.
identifier_types = {c: 'string' for c in
                    ['hProperty', 'hUnit', 'hBuilding', 'sPropCode', 'sUnitCode', 'sBuilding', 'sSite']}
data = {name: pd.read_csv(path, dtype=identifier_types) for name, path in FILES.items()}
expected_shapes = {'listings': (1061, 29), 'lease_history': (4302, 35),
                   'concessions': (3560, 18), 'asking_history': (3857, 14)}
display(pd.DataFrame([{'dataset': name, 'rows': len(df), 'columns': len(df.columns),
                      'matches_challenge_dimensions': df.shape == expected_shapes[name]}
                     for name, df in data.items()]))

,jeu de données,lignes,colonnes,dimensions conformes au défi
0,listings,1061,29,True
1,lease_history,4302,35,True
2,concessions,3560,18,True
3,asking_history,3857,14,True


## Schéma, valeurs manquantes, doublons et couverture

Les dtypes ci-dessous sont ceux du chargement (les identifiants utilisent explicitement des chaînes). La conversion des dates est temporaire et ne remplace pas les colonnes sources. Les valeurs manquantes suivent les conventions CSV par défaut de pandas ; les valeurs composées uniquement d’espaces sont vérifiées séparément. Les dénombrements d’identifiants uniques excluent les valeurs manquantes. Les bâtiments sont comptés par identifiant et par paire propriété/bâtiment, car les libellés peuvent se répéter entre propriétés.

In [2]:
def key_pairs(df, columns=('sPropCode', 'sUnitCode')):
    return df.loc[df[list(columns)].notna().all(axis=1), list(columns)]

def categorical(df, columns):
    for col in columns:
        if col in df:
            print(col)
            display(df[col].fillna('<MANQUANT>').value_counts(dropna=False)
                    .rename_axis('category').to_frame('count'))

def numeric(df, columns):
    result = []
    for col in columns:
        if col in df:
            values = pd.to_numeric(df[col], errors='coerce')
            stats = values.describe(percentiles=[.25, .5, .75]).to_dict()
            stats.update(column=col, nonmissing_unparseable=int((df[col].notna() & values.isna()).sum()),
                         zero_count=int(values.eq(0).sum()), negative_count=int(values.lt(0).sum()))
            result.append(stats)
    if result:
        display(pd.DataFrame(result).set_index('column'))

for name, df in data.items():
    display(Markdown(f'### {name}'))
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing_count': df.isna().sum(),
                          'missing_pct': df.isna().mean().mul(100).round(2),
                          'whitespace_only_count': [df[c].astype('string').str.fullmatch(r'\s*').fillna(False).sum()
                                                    for c in df.columns]}))
    print('Lignes exactement identiques au-delà de la première occurrence :', int(df.duplicated().sum()))
    ids = [c for c in identifier_types if c in df]
    display(df[ids].nunique().rename('unique_nonmissing').to_frame())
    for building in ['hBuilding', 'sBuilding']:
        print(f'Paires (sPropCode, {building}) uniques :', len(key_pairs(df, ('sPropCode', building)).drop_duplicates()))
    print('Paires officielles d’unité complètes et distinctes :', len(key_pairs(df).drop_duplicates()))

### listings

,dtype,valeurs manquantes,valeurs manquantes (%),valeurs composées d’espaces
hProperty,string,0,0.0,0
hUnit,string,0,0.0,0
hBuilding,string,0,0.0,0
sPropCode,string,0,0.0,0
sUnitCode,string,0,0.0,0
sBuilding,string,0,0.0,0
sCity,object,0,0.0,0
sState,object,0,0.0,0
sAddr1,object,0,0.0,0
sLatitude,float64,0,0.0,0


Lignes exactement identiques au-delà de la première occurrence : 0


,valeurs uniques non manquantes
hProperty,9
hUnit,1061
hBuilding,6
sPropCode,9
sUnitCode,519
sBuilding,6
sSite,6


Paires (sPropCode, hBuilding) uniques : 9
Paires (sPropCode, sBuilding) uniques : 9
Paires officielles d’unité complètes et distinctes : 1061


...

### asking_history

,dtype,valeurs manquantes,valeurs manquantes (%),valeurs composées d’espaces
hProperty,string,0,0.0,0
hUnit,string,0,0.0,0
hBuilding,string,0,0.0,0
sPropCode,string,0,0.0,0
sUnitCode,string,0,0.0,0
sBuilding,string,0,0.0,0
sSite,string,0,0.0,0
sCity,object,0,0.0,0
sState,object,0,0.0,0
sBeds,int64,0,0.0,0


Lignes exactement identiques au-delà de la première occurrence : 0


,valeurs uniques non manquantes
hProperty,9
hUnit,1061
hBuilding,6
sPropCode,9
sUnitCode,519
sBuilding,6
sSite,6


Paires (sPropCode, hBuilding) uniques : 9
Paires (sPropCode, sBuilding) uniques : 9
Paires officielles d’unité complètes et distinctes : 1061


## Colonnes attendues et dates

Les dates probables sont repérées à partir des noms de colonnes, notamment `sMonth`, `sAsOf` et `sAvailable`. La couverture et les années disponibles concernent uniquement les valeurs converties avec succès. Les échecs de conversion et les valeurs manquantes sont comptés explicitement. Un nom évoquant une date ne suffit pas à établir sa signification métier.

In [3]:
expected_columns = {
    'listings': ['sPropCode', 'sUnitCode', 'sSite', 'sBeds', 'sBaths', 'sSqft', 'sFloor', 'sUnitType', 'sRent'],
    'lease_history': ['sPropCode', 'sUnitCode', 'sRent', 'sRentEffective', 'sConcession', 'sSignDate',
                      'sLeaseFrom', 'sLeaseTo', 'sTermMonths', 'sTermSeq', 'sRenewal'],
    'concessions': ['sPropCode', 'sUnitCode', 'sChargeCode', 'sChargeDesc', 'sAmount', 'sDateFrom', 'sDateTo', 'sMonths'],
    'asking_history': ['sPropCode', 'sUnitCode', 'sMonth', 'sAskingRent']}
date_rows = []
parsed_dates = {}
for name, df in data.items():
    print(name, 'colonnes attendues absentes :', [c for c in expected_columns[name] if c not in df])
    candidates = [c for c in df if 'date' in c.lower() or c in
                  ['sMonth', 'sAsOf', 'sAvailable', 'sLeaseFrom', 'sLeaseTo']]
    print('Colonnes de date probables :', candidates)
    for col in candidates:
        parsed = pd.to_datetime(df[col], errors='coerce', format='mixed')
        parsed_dates[(name, col)] = parsed
        date_rows.append({'dataset': name, 'column': col, 'missing': int(df[col].isna().sum()),
                          'unparseable_nonmissing': int((df[col].notna() & parsed.isna()).sum()),
                          'min': str(parsed.min()), 'max': str(parsed.max()),
                          'years': sorted(parsed.dt.year.dropna().astype(int).unique().tolist())})
date_summary = pd.DataFrame(date_rows)
display(date_summary)

listings colonnes attendues absentes : []
Colonnes de date probables : ['sAsOf', 'sAvailable']
lease_history colonnes attendues absentes : []
Colonnes de date probables : ['sAvailable', 'sSignDate', 'sLeaseFrom', 'sLeaseTo']
concessions colonnes attendues absentes : []
Colonnes de date probables : ['sDateFrom', 'sDateTo']
asking_history colonnes attendues absentes : []
Colonnes de date probables : ['sMonth']


,jeu de données,colonne,valeurs manquantes,échecs de conversion non manquants,min,max,années
0,listings,sAsOf,0,0,2025-01-01 00:00:00,2025-12-01 00:00:00,[2025]
1,listings,sAvailable,0,0,2025-12-31 00:00:00,2026-09-30 00:00:00,"[2025, 2026]"
2,lease_history,sAvailable,0,0,2017-01-01 00:00:00,2025-12-31 00:00:00,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 202..."
3,lease_history,sSignDate,0,0,2016-10-26 00:00:00,2025-12-27 00:00:00,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."
4,lease_history,sLeaseFrom,0,0,2017-01-01 00:00:00,2025-12-31 00:00:00,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 202..."
5,lease_history,sLeaseTo,0,0,2017-09-29 00:00:00,2027-11-05 00:00:00,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 202..."
6,concessions,sDateFrom,0,0,2017-01-01 00:00:00,2027-03-01 00:00:00,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 202..."
7,concessions,sDateTo,0,0,2017-08-29 00:00:00,2027-05-01 00:00:00,"[2017, 2018, 2019, 2020, 2021, 2022, 2023, 202..."
8,asking_history,sMonth,0,0,2016-10-01 00:00:00,2025-12-01 00:00:00,"[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202..."


## Validation de l’identité officielle des unités

Les lignes en doublon au-delà de la première occurrence et les lignes appartenant aux groupes de doublons sont des mesures distinctes. Le test alternatif site/unité compte les groupes correspondant à plusieurs paires officielles propriété/unité. Il indique aussi les paires officielles excédentaires et les lignes concernées. Les observations répétées d’une même unité officielle ne constituent pas, à elles seules, une collision de la clé site/unité. La clé officielle reste inchangée.

In [4]:
identity_rows = []
for name, df in data.items():
    official = key_pairs(df)
    complete = df.dropna(subset=['sPropCode', 'sUnitCode', 'sSite'])
    mapping = complete[['sSite', 'sUnitCode', 'sPropCode']].drop_duplicates()
    counts = mapping.groupby(['sSite', 'sUnitCode'], dropna=False).size()
    collided = counts[counts > 1]
    affected = complete.set_index(['sSite', 'sUnitCode']).index.isin(collided.index).sum()
    identity_rows.append({'dataset': name, 'missing_official_key_rows': len(df)-len(official),
        'unique_UNIT_KEY': len(official.drop_duplicates()),
        'duplicate_UNIT_KEY_excess_rows': int(official.duplicated().sum()),
        'rows_in_duplicate_UNIT_KEY_groups': int(official.duplicated(keep=False).sum()),
        'missing_site_test_components': len(df)-len(complete),
        'colliding_site_unit_groups': len(collided),
        'excess_official_pairs_under_site_key': int((collided-1).sum()),
        'rows_affected_by_site_collisions': int(affected)})
identity_summary = pd.DataFrame(identity_rows).set_index('dataset')
display(identity_summary)

,lignes à clé officielle incomplète,UNIT_KEY uniques,lignes excédentaires de UNIT_KEY dupliquée,lignes dans les groupes de UNIT_KEY dupliquée,lignes à composantes site/unité manquantes,groupes site/unité en collision,paires officielles excédentaires sous la clé site,lignes concernées par les collisions site
jeu de données,,,,,,,,
listings,0,1061,0,0,0,105,143,248
lease_history,0,1061,3241,4133,0,105,143,1332
concessions,0,1014,2546,3391,0,99,125,918
asking_history,0,1061,2796,3646,0,105,143,1156


## Listings : caractéristiques observées des unités

In [5]:
df = data['listings']
numeric(df, ['sBeds', 'sBaths', 'sSqft', 'sFloor', 'sRent'])
categorical(df, ['sBeds', 'sBaths', 'sFloor', 'sUnitType', 'sUnitSubtype', 'sPropType', 'sFurnishing', 'sSmoking', 'sCats', 'sDogs'])
print('Colonne de loyer présente : sRent. Son rôle de loyer demandé doit être confirmé dans la documentation.')

,effectif,moyenne,écart type,min,25%,50%,75%,max,valeurs non manquantes non convertibles,valeurs nulles,valeurs négatives
colonne,,,,,,,,,,,
sBeds,1061.0,1.599434,0.613342,0.0,1.0,2.0,2.0,3.0,0,49,0
sBaths,1061.0,1.285580,0.453986,1.0,1.0,1.0,2.0,3.0,0,0,0
sSqft,1061.0,974.311970,317.052311,445.0,640.0,1050.0,1260.0,1905.0,0,0,0
sFloor,1061.0,9.315740,6.225270,1.0,4.0,7.0,14.0,28.0,0,0,0
sRent,1061.0,2313.812441,862.658273,740.0,1525.0,2370.0,2725.0,7160.0,0,0,0


sBeds


,effectif
catégorie,
2,639
1,350
0,49
3,23


sBaths


,effectif
catégorie,
1.0,759
2.0,301
3.0,1


...

sCats


,effectif
catégorie,
True,1061


sDogs


,effectif
catégorie,
True,1061


Colonne de loyer présente : sRent. Son rôle de loyer demandé doit être confirmé dans la documentation.


## Historique des baux : loyers, concessions, durées, renouvellements et dates

In [6]:
df = data['lease_history']
numeric(df, ['sRent', 'sRentEffective', 'sConcession', 'sTermMonths', 'sTermSeq'])
categorical(df, ['sRenewal', 'sTermMonths', 'sTermSeq'])
print('La couverture des propriétés/unités figure ci-dessus ; les plages et années des baux figurent dans le résumé des dates.')
for start, end in [('sLeaseFrom', 'sLeaseTo')]:
    a, b = parsed_dates[('lease_history', start)], parsed_dates[('lease_history', end)]
    print('Fin de bail antérieure au début, avec deux dates valides :', int((b < a).sum()))

,effectif,moyenne,écart type,min,25%,50%,75%,max,valeurs non manquantes non convertibles,valeurs nulles,valeurs négatives
colonne,,,,,,,,,,,
sRent,4302.0,1999.192236,744.203387,600.00,1405.0,1970.000,2400.000,6835.0,0,0,0
sRentEffective,4302.0,1896.919040,685.722634,589.13,1340.0,1893.205,2259.045,6310.0,0,0,0
sConcession,4302.0,0.632032,0.482309,0.00,0.0,1.000,1.000,1.0,0,1583,0
sTermMonths,4302.0,12.202348,1.882098,5.90,12.0,12.000,12.000,23.9,0,0,0
sTermSeq,4302.0,3.308926,2.100721,1.00,2.0,3.000,5.000,8.0,0,0,0


sRenewal


,effectif
catégorie,
0,2337
1,1965


sTermMonths


,effectif
catégorie,
12.0,4067
17.9,93
5.9,76
23.9,66


sTermSeq


,effectif
catégorie,
1,1061
2,892
3,691
4,446
5,374
6,366
7,309
8,163


La couverture des propriétés/unités figure ci-dessus ; les plages et années des baux figurent dans le résumé des dates.
Fin de bail antérieure au début, avec deux dates valides : 0


## Concessions : catégories et montants observés

Aucune interprétation récurrente ou mensuelle n’est attribuée à `sAmount` ou `sMonths`.

In [7]:
df = data['concessions']
categorical(df, ['sChargeCode', 'sChargeDesc'])
numeric(df, ['sAmount', 'sMonths'])
print('Fin de concession antérieure au début :', int((parsed_dates[('concessions', 'sDateTo')] < parsed_dates[('concessions', 'sDateFrom')]).sum()))

sChargeCode


,effectif
catégorie,
PromoPay,1494
freepark,1006
Freeothe,570
freelock,175
freerent,166
Indem,69
freeappl,44
Referenc,36


sChargeDesc


,effectif
catégorie,
Promo - Payable in the future,1494
Free Parking,1006
Free other,570
Free Locker,175
Free rent,166
Client indemnity,69
Free Appliances,44
Referencing program,36


,effectif,moyenne,écart type,min,25%,50%,75%,max,valeurs non manquantes non convertibles,valeurs nulles,valeurs négatives
colonne,,,,,,,,,,,
sAmount,3560.0,-782.034994,997.971793,-10927.95,-1150.71,-310.775,-117.175,-1.79,0,0,3560
sMonths,3560.0,5.703090,4.792554,1.00,1.00,3.000,12.000,23.00,0,0,0


Fin de concession antérieure au début : 0


## Historique des loyers demandés : couverture et fréquence des observations

La fréquence est mesurée au sein des unités officielles à partir des dates valides distinctes. L’espacement en mois calendaires est un diagnostic ; il ne prouve pas une couverture mensuelle complète. Seules les distributions des intervalles sont affichées.

In [8]:
df = data['asking_history']
numeric(df, ['sAskingRent'])
dates = parsed_dates[('asking_history', 'sMonth')]
print('Propriétés :', df['sPropCode'].nunique(), 'unités officielles :', len(key_pairs(df).drop_duplicates()))
print('Nombre de dates valides :', dates.notna().sum(), 'dates distinctes :', dates.nunique())
frequency = df[['sPropCode', 'sUnitCode']].copy()
frequency['date'] = dates
frequency = frequency.dropna()
print('Lignes excédentaires par paire unité officielle/date :', int(frequency.duplicated().sum()))
frequency = frequency.drop_duplicates().sort_values(['sPropCode', 'sUnitCode', 'date'])
frequency['month_index'] = frequency['date'].dt.year * 12 + frequency['date'].dt.month
intervals = frequency.groupby(['sPropCode', 'sUnitCode'])['date'].diff().dt.total_seconds().div(86400).dropna()
month_intervals = frequency.groupby(['sPropCode', 'sUnitCode'])['month_index'].diff().dropna()
display(intervals.value_counts().sort_index().rename_axis('days_between_observations').to_frame('count'))
display(month_intervals.value_counts().sort_index().rename_axis('calendar_month_gap').to_frame('count'))
display(frequency.groupby(['sPropCode', 'sUnitCode']).size().describe().rename('observations_per_unit').to_frame())
display(dates.dt.year.value_counts().sort_index().rename_axis('year').to_frame('observations'))
print('Jours du mois observés :', sorted(dates.dt.day.dropna().astype(int).unique().tolist()))

,effectif,moyenne,écart type,min,25%,50%,75%,max,valeurs non manquantes non convertibles,valeurs nulles,valeurs négatives
colonne,,,,,,,,,,,
sAskingRent,3857.0,2036.845994,766.76129,605.0,1425.0,2005.0,2445.0,7135.0,0,0,0


Propriétés : 9 unités officielles : 1061
Nombre de dates valides : 3857 dates distinctes : 111
Lignes excédentaires par paire unité officielle/date : 0


,effectif
jours entre observations,
28.0,32
29.0,12
30.0,229
31.0,374
59.0,5
...,...
1858.0,1
1888.0,1
2131.0,1


,effectif
écart en mois calendaires,
1.0,647
2.0,37
4.0,4
5.0,8
6.0,10
7.0,5
8.0,5
9.0,19
10.0,130


,observations par unité
count,1061.000000
mean,3.635250
std,2.339941
min,1.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,12.000000


,observations
année,
2016,16
2017,118
2018,205
2019,328
2020,303
2021,290
2022,392
2023,663
2024,773


Jours du mois observés : [1]


## Vérification de l’intégrité

In [9]:
assert all(hashlib.sha256(path.read_bytes()).hexdigest() == raw_hashes[name]
           for name, path in FILES.items()), 'Un fichier brut a changé pendant l’audit'
print('Les empreintes des quatre fichiers bruts sont inchangées. Ce notebook n’exporte aucun jeu de données.')

Les empreintes des quatre fichiers bruts sont inchangées. Ce notebook n’exporte aucun jeu de données.


La cellule suivante régénère les constats directement à partir des résultats agrégés.

In [10]:
facts = []
issues = []
for name, df in data.items():
    facts.append(f"**{name}**: {len(df):,} lignes × {len(df.columns)} colonnes ; "
                 f"{df['sPropCode'].nunique()} propriétés, {df['hBuilding'].nunique()} valeurs hBuilding distinctes, "
                 f"{len(key_pairs(df, ('sPropCode', 'hBuilding')).drop_duplicates())} paires propriété/bâtiment, "
                 f"{len(key_pairs(df).drop_duplicates()):,} unités officielles ; "
                 f"{int(df.duplicated().sum())} lignes exactement identiques au-delà de la première occurrence.")
    missing = df.isna().sum()
    facts.append(f"{name} valeurs manquantes : " + ('; '.join(f'{c}: {int(v):,} ({v/len(df):.1%})' for c, v in missing.items() if v) or 'aucune valeur manquante détectée par pandas') + '.')
for row in date_rows:
    facts.append(f"{row['dataset']}.{row['column']}: {row['min']} à {row['max']} ; années {row['years']}; {row['unparseable_nonmissing']} échecs de conversion parmi les valeurs non manquantes.")
facts.append('Les colonnes de clé officielle sPropCode et sUnitCode existent dans tous les jeux. Toutes les colonnes attendues listées existent : ' + str(all(set(cols).issubset(data[name].columns) for name, cols in expected_columns.items())) + '.')
for name, row in identity_summary.iterrows():
    facts.append(f"{name}: {int(row['unique_UNIT_KEY']):,} paires UNIT_KEY distinctes, {int(row['duplicate_UNIT_KEY_excess_rows']):,} lignes excédentaires de clé dupliquée, {int(row['missing_official_key_rows'])} clés incomplètes. Test site/unité : {int(row['colliding_site_unit_groups'])} groupes en collision, {int(row['excess_official_pairs_under_site_key'])} paires officielles excédentaires, {int(row['rows_affected_by_site_collisions'])} lignes concernées.")
for name, cols in {'listings': ['sUnitType', 'sUnitSubtype', 'sBeds', 'sBaths'], 'lease_history': ['sRenewal'], 'concessions': ['sChargeCode', 'sChargeDesc']}.items():
    for col in cols:
        counts = data[name][col].fillna('<MANQUANT>').value_counts()
        facts.append(f"{name}.{col} catégories (effectif) : " + '; '.join(f'{value}: {count}' for value, count in counts.items()) + '.')
facts.append('Écarts en mois calendaires de l’historique des loyers demandés (effectif) : ' + '; '.join(f'{gap:g}: {count}' for gap, count in month_intervals.value_counts().sort_index().items()) + '. Ces intervalles observés ne garantissent pas une couverture mensuelle complète.')
facts.append('Les 3,560 montants de concession sont négatifs (de -10,927.95 à -1.79) ; sMonths varie de 1 à 23. Dans les baux, sConcession contient uniquement 0 et 1 ; leur signification n’est pas déduite.')
facts.append('Valeurs et effectifs de sTermMonths dans les baux : 12.0 (4,067), 17.9 (93), 5.9 (76), 23.9 (66). Aucun intervalle début/fin inversé n’a été observé pour les baux ou concessions.')
facts.append('Les observations de loyers demandés sont toutes datées du premier jour du mois ; elles comptent 111 dates distinctes, aucune paire unité officielle/date dupliquée et 1–12 observations par unité.')
issues.extend([
    'Confirmer les conventions de signe des concessions et les durées de bail non entières (5.9, 17.9, 23.9) ; ces valeurs observées ne constituent pas automatiquement des erreurs.',
    'Examiner pourquoi l’historique des loyers demandés est peu dense par unité malgré des dates alignées sur les mois ; aucune interpolation ni hypothèse de couverture complète n’est faite.',
    'Clarifier la relation entre 9 codes de propriété, 6 identifiants de bâtiment et 9 paires propriété/bâtiment avant d’interpréter le nombre de bâtiments physiques.',
    'Examiner les valeurs manquantes, échecs de conversion des dates, valeurs non numériques, montants négatifs/nuls et intervalles inversés signalés ci-dessus avant les phases suivantes ; le signe d’un nombre ne suffit pas à établir une erreur.',
    'Examiner les clés officielles dupliquées dans listings, le cas échéant. Les clés répétées dans les historiques peuvent correspondre à des observations répétées ; leur granularité métier doit être confirmée.',
    'Confirmer dans la documentation la signification de sRent dans listings, les codes de séquence et de renouvellement, ainsi que la sémantique montant/période des concessions. Aucun comportement mensuel ou récurrent n’est déduit.',
    'Confirmer la signification des identifiants de bâtiment et la couverture historique. Les dates extrêmes et les effectifs par année ne prouvent pas une couverture complète.',
    'Le test de collision site/unité sert uniquement à la validation ; conserver la clé officielle propriété/unité quel que soit le résultat.'
])
findings_markdown = '## Constats de l’audit des données\n\n### FAITS CONFIRMÉS\n\n' + '\n'.join('- ' + f for f in facts) + '\n\n### QUESTIONS / POINTS À INVESTIGUER\n\n' + '\n'.join('- ' + q for q in issues)
display(Markdown(findings_markdown))

## Constats de l’audit des données

### FAITS CONFIRMÉS

- **listings**: 1,061 lignes × 29 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,061 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- listings valeurs manquantes : aucune valeur manquante détectée par pandas.
- **lease_history**: 4,302 lignes × 35 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,061 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- lease_history valeurs manquantes : aucune valeur manquante détectée par pandas.
- **concessions**: 3,560 lignes × 18 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,014 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- concessions valeurs manquantes : aucune valeur manquante détectée par pandas.
- **asking_history**: 3,857 lignes × 14 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,061 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- asking_history valeurs manquantes : aucune valeur manquante détectée par pandas.
- listings.sAsOf: 2025-01-01 00:00:00 à 2025-12-01 00:00:00 ; années [2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- listings.sAvailable: 2025-12-31 00:00:00 à 2026-09-30 00:00:00 ; années [2025, 2026]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sAvailable: 2017-01-01 00:00:00 à 2025-12-31 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sSignDate: 2016-10-26 00:00:00 à 2025-12-27 00:00:00 ; années [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sLeaseFrom: 2017-01-01 00:00:00 à 2025-12-31 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sLeaseTo: 2017-09-29 00:00:00 à 2027-11-05 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]; 0 échecs de conversion parmi les valeurs non manquantes.
- concessions.sDateFrom: 2017-01-01 00:00:00 à 2027-03-01 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]; 0 échecs de conversion parmi les valeurs non manquantes.
- concessions.sDateTo: 2017-08-29 00:00:00 à 2027-05-01 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]; 0 échecs de conversion parmi les valeurs non manquantes.
- asking_history.sMonth: 2016-10-01 00:00:00 à 2025-12-01 00:00:00 ; années [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- Les colonnes de clé officielle sPropCode et sUnitCode existent dans tous les jeux. Toutes les colonnes attendues listées existent : True.
- listings: 1,061 paires UNIT_KEY distinctes, 0 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 105 groupes en collision, 143 paires officielles excédentaires, 248 lignes concernées.
- lease_history: 1,061 paires UNIT_KEY distinctes, 3,241 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 105 groupes en collision, 143 paires officielles excédentaires, 1332 lignes concernées.
- concessions: 1,014 paires UNIT_KEY distinctes, 2,546 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 99 groupes en collision, 125 paires officielles excédentaires, 918 lignes concernées.
- asking_history: 1,061 paires UNIT_KEY distinctes, 2,796 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 105 groupes en collision, 143 paires officielles excédentaires, 1156 lignes concernées.
- listings.sUnitType catégories (effectif) : 2_1: 343; 1_1: 262; 2_2: 257; 1_1d: 88; 0_1: 49; 2_2d: 24; 3_2: 16; 2_1d: 15; 3_2d: 4; 3_1: 2; 3_3: 1.
- listings.sUnitSubtype catégories (effectif) : 2 Bed: 639; 1 Bed: 350; Studio: 49; 3 Bed: 23.
- listings.sBeds catégories (effectif) : 2: 639; 1: 350; 0: 49; 3: 23.
- listings.sBaths catégories (effectif) : 1.0: 759; 2.0: 301; 3.0: 1.
- lease_history.sRenewal catégories (effectif) : 0: 2337; 1: 1965.
- concessions.sChargeCode catégories (effectif) : PromoPay: 1494; freepark: 1006; Freeothe: 570; freelock: 175; freerent: 166; Indem: 69; freeappl: 44; Referenc: 36.
- concessions.sChargeDesc catégories (effectif) : Promo - Payable in the future: 1494; Free Parking: 1006; Free other: 570; Free Locker: 175; Free rent: 166; Client indemnity: 69; Free Appliances: 44; Referencing program: 36.
- Écarts en mois calendaires de l’historique des loyers demandés (effectif) : 1: 647; 2: 37; 4: 4; 5: 8; 6: 10; 7: 5; 8: 5; 9: 19; 10: 130; 11: 357; 12: 507; 13: 324; 14: 123; 15: 26; 16: 11; 17: 10; 18: 13; 19: 13; 20: 8; 21: 1; 22: 24; 23: 79; 24: 124; 25: 95; 26: 40; 27: 9; 28: 6; 29: 3; 30: 6; 31: 2; 32: 5; 33: 2; 34: 8; 35: 20; 36: 38; 37: 14; 38: 17; 39: 2; 40: 4; 42: 2; 43: 1; 44: 2; 45: 2; 47: 3; 48: 9; 49: 4; 50: 4; 51: 1; 52: 1; 53: 1; 59: 1; 60: 3; 61: 2; 62: 1; 70: 1; 71: 1; 73: 1. Ces intervalles observés ne garantissent pas une couverture mensuelle complète.
- Les 3,560 montants de concession sont négatifs (de -10,927.95 à -1.79) ; sMonths varie de 1 à 23. Dans les baux, sConcession contient uniquement 0 et 1 ; leur signification n’est pas déduite.
- Valeurs et effectifs de sTermMonths dans les baux : 12.0 (4,067), 17.9 (93), 5.9 (76), 23.9 (66). Aucun intervalle début/fin inversé n’a été observé pour les baux ou concessions.
- Les observations de loyers demandés sont toutes datées du premier jour du mois ; elles comptent 111 dates distinctes, aucune paire unité officielle/date dupliquée et 1–12 observations par unité.

### QUESTIONS / POINTS À INVESTIGUER

- Confirmer les conventions de signe des concessions et les durées de bail non entières (5.9, 17.9, 23.9) ; ces valeurs observées ne constituent pas automatiquement des erreurs.
- Examiner pourquoi l’historique des loyers demandés est peu dense par unité malgré des dates alignées sur les mois ; aucune interpolation ni hypothèse de couverture complète n’est faite.
- Clarifier la relation entre 9 codes de propriété, 6 identifiants de bâtiment et 9 paires propriété/bâtiment avant d’interpréter le nombre de bâtiments physiques.
- Examiner les valeurs manquantes, échecs de conversion des dates, valeurs non numériques, montants négatifs/nuls et intervalles inversés signalés ci-dessus avant les phases suivantes ; le signe d’un nombre ne suffit pas à établir une erreur.
- Examiner les clés officielles dupliquées dans listings, le cas échéant. Les clés répétées dans les historiques peuvent correspondre à des observations répétées ; leur granularité métier doit être confirmée.
- Confirmer dans la documentation la signification de sRent dans listings, les codes de séquence et de renouvellement, ainsi que la sémantique montant/période des concessions. Aucun comportement mensuel ou récurrent n’est déduit.
- Confirmer la signification des identifiants de bâtiment et la couverture historique. Les dates extrêmes et les effectifs par année ne prouvent pas une couverture complète.
- Le test de collision site/unité sert uniquement à la validation ; conserver la clé officielle propriété/unité quel que soit le résultat.

## Constats de l’audit des données

### FAITS CONFIRMÉS

- **listings**: 1,061 lignes × 29 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,061 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- listings valeurs manquantes : aucune valeur manquante détectée par pandas.
- **lease_history**: 4,302 lignes × 35 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,061 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- lease_history valeurs manquantes : aucune valeur manquante détectée par pandas.
- **concessions**: 3,560 lignes × 18 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,014 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- concessions valeurs manquantes : aucune valeur manquante détectée par pandas.
- **asking_history**: 3,857 lignes × 14 colonnes ; 9 propriétés, 6 valeurs hBuilding distinctes, 9 paires propriété/bâtiment, 1,061 unités officielles ; 0 lignes exactement identiques au-delà de la première occurrence.
- asking_history valeurs manquantes : aucune valeur manquante détectée par pandas.
- listings.sAsOf: 2025-01-01 00:00:00 à 2025-12-01 00:00:00 ; années [2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- listings.sAvailable: 2025-12-31 00:00:00 à 2026-09-30 00:00:00 ; années [2025, 2026]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sAvailable: 2017-01-01 00:00:00 à 2025-12-31 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sSignDate: 2016-10-26 00:00:00 à 2025-12-27 00:00:00 ; années [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sLeaseFrom: 2017-01-01 00:00:00 à 2025-12-31 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- lease_history.sLeaseTo: 2017-09-29 00:00:00 à 2027-11-05 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]; 0 échecs de conversion parmi les valeurs non manquantes.
- concessions.sDateFrom: 2017-01-01 00:00:00 à 2027-03-01 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]; 0 échecs de conversion parmi les valeurs non manquantes.
- concessions.sDateTo: 2017-08-29 00:00:00 à 2027-05-01 00:00:00 ; années [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]; 0 échecs de conversion parmi les valeurs non manquantes.
- asking_history.sMonth: 2016-10-01 00:00:00 à 2025-12-01 00:00:00 ; années [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]; 0 échecs de conversion parmi les valeurs non manquantes.
- Les colonnes de clé officielle sPropCode et sUnitCode existent dans tous les jeux. Toutes les colonnes attendues listées existent : True.
- listings: 1,061 paires UNIT_KEY distinctes, 0 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 105 groupes en collision, 143 paires officielles excédentaires, 248 lignes concernées.
- lease_history: 1,061 paires UNIT_KEY distinctes, 3,241 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 105 groupes en collision, 143 paires officielles excédentaires, 1332 lignes concernées.
- concessions: 1,014 paires UNIT_KEY distinctes, 2,546 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 99 groupes en collision, 125 paires officielles excédentaires, 918 lignes concernées.
- asking_history: 1,061 paires UNIT_KEY distinctes, 2,796 lignes excédentaires de clé dupliquée, 0 clés incomplètes. Test site/unité : 105 groupes en collision, 143 paires officielles excédentaires, 1156 lignes concernées.
- listings.sUnitType catégories (effectif) : 2_1: 343; 1_1: 262; 2_2: 257; 1_1d: 88; 0_1: 49; 2_2d: 24; 3_2: 16; 2_1d: 15; 3_2d: 4; 3_1: 2; 3_3: 1.
- listings.sUnitSubtype catégories (effectif) : 2 Bed: 639; 1 Bed: 350; Studio: 49; 3 Bed: 23.
- listings.sBeds catégories (effectif) : 2: 639; 1: 350; 0: 49; 3: 23.
- listings.sBaths catégories (effectif) : 1.0: 759; 2.0: 301; 3.0: 1.
- lease_history.sRenewal catégories (effectif) : 0: 2337; 1: 1965.
- concessions.sChargeCode catégories (effectif) : PromoPay: 1494; freepark: 1006; Freeothe: 570; freelock: 175; freerent: 166; Indem: 69; freeappl: 44; Referenc: 36.
- concessions.sChargeDesc catégories (effectif) : Promo - Payable in the future: 1494; Free Parking: 1006; Free other: 570; Free Locker: 175; Free rent: 166; Client indemnity: 69; Free Appliances: 44; Referencing program: 36.
- Écarts en mois calendaires de l’historique des loyers demandés (effectif) : 1: 647; 2: 37; 4: 4; 5: 8; 6: 10; 7: 5; 8: 5; 9: 19; 10: 130; 11: 357; 12: 507; 13: 324; 14: 123; 15: 26; 16: 11; 17: 10; 18: 13; 19: 13; 20: 8; 21: 1; 22: 24; 23: 79; 24: 124; 25: 95; 26: 40; 27: 9; 28: 6; 29: 3; 30: 6; 31: 2; 32: 5; 33: 2; 34: 8; 35: 20; 36: 38; 37: 14; 38: 17; 39: 2; 40: 4; 42: 2; 43: 1; 44: 2; 45: 2; 47: 3; 48: 9; 49: 4; 50: 4; 51: 1; 52: 1; 53: 1; 59: 1; 60: 3; 61: 2; 62: 1; 70: 1; 71: 1; 73: 1. Ces intervalles observés ne garantissent pas une couverture mensuelle complète.
- Les 3,560 montants de concession sont négatifs (de -10,927.95 à -1.79) ; sMonths varie de 1 à 23. Dans les baux, sConcession contient uniquement 0 et 1 ; leur signification n’est pas déduite.
- Valeurs et effectifs de sTermMonths dans les baux : 12.0 (4,067), 17.9 (93), 5.9 (76), 23.9 (66). Aucun intervalle début/fin inversé n’a été observé pour les baux ou concessions.
- Les observations de loyers demandés sont toutes datées du premier jour du mois ; elles comptent 111 dates distinctes, aucune paire unité officielle/date dupliquée et 1–12 observations par unité.

### QUESTIONS / POINTS À INVESTIGUER

- Confirmer les conventions de signe des concessions et les durées de bail non entières (5.9, 17.9, 23.9) ; ces valeurs observées ne constituent pas automatiquement des erreurs.
- Examiner pourquoi l’historique des loyers demandés est peu dense par unité malgré des dates alignées sur les mois ; aucune interpolation ni hypothèse de couverture complète n’est faite.
- Clarifier la relation entre 9 codes de propriété, 6 identifiants de bâtiment et 9 paires propriété/bâtiment avant d’interpréter le nombre de bâtiments physiques.
- Examiner les valeurs manquantes, échecs de conversion des dates, valeurs non numériques, montants négatifs/nuls et intervalles inversés signalés ci-dessus avant les phases suivantes ; le signe d’un nombre ne suffit pas à établir une erreur.
- Examiner les clés officielles dupliquées dans listings, le cas échéant. Les clés répétées dans les historiques peuvent correspondre à des observations répétées ; leur granularité métier doit être confirmée.
- Confirmer dans la documentation la signification de sRent dans listings, les codes de séquence et de renouvellement, ainsi que la sémantique montant/période des concessions. Aucun comportement mensuel ou récurrent n’est déduit.
- Confirmer la signification des identifiants de bâtiment et la couverture historique. Les dates extrêmes et les effectifs par année ne prouvent pas une couverture complète.
- Le test de collision site/unité sert uniquement à la validation ; conserver la clé officielle propriété/unité quel que soit le résultat.